# 05 — Analysis: Ablations, Noise Robustness & Representation Analysis

Notebook tổng hợp phân tích sâu cho paper:
1. E6 — Qubit scaling
2. E7 — Circuit depth scaling
3. E9 — NISQ noise robustness
4. t-SNE fused representations (quantum vs classical)
5. Parameter efficiency + circuit analysis

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"
FIG_DIR = PROJECT_ROOT / "paper" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path):
    p = Path(path)
    if not p.exists():
        print(f"[missing] {p}")
        return {}
    with open(p) as f:
        return json.load(f)


ablation = load_json(RESULTS_DIR / "ablation_results.json")
main_results = load_json(RESULTS_DIR / "results.json")
print("Ablation keys :", list(ablation.keys()))
print("Main keys     :", list(main_results.keys()))

## 1. E6 — Qubit scaling

In [ ]:
e6 = ablation.get("E6") or main_results.get("E6_qubit_scaling") or {}
if e6:
    qubits = sorted(int(k) for k in e6.keys())
    accs = [e6[str(q)]["accuracy"] for q in qubits]
    plt.figure(figsize=(8, 5))
    plt.plot(qubits, accs, "o-", linewidth=2, markersize=9, color="#2196F3", label="QFL-Tensor")
    plt.xlabel("Number of Qubits")
    plt.ylabel("Test Accuracy")
    plt.title("Quantum Resource Scaling: Accuracy vs Qubits")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "qubit_scaling.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Chưa có dữ liệu E6 — chạy: python experiments/run_ablation.py --experiment E6")

## 2. E7 — Circuit depth scaling

In [ ]:
e7 = ablation.get("E7") or main_results.get("E7_depth_scaling") or {}
if e7:
    depths = sorted(int(k) for k in e7.keys())
    accs = [e7[str(d)]["accuracy"] for d in depths]
    plt.figure(figsize=(8, 5))
    plt.plot(depths, accs, "s-", linewidth=2, markersize=9, color="#9C27B0", label="QFL-Tensor")
    plt.xlabel("PQC Depth (layers)")
    plt.ylabel("Test Accuracy")
    plt.title("Circuit Depth Effect on Performance")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "depth_scaling.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Chưa có dữ liệu E7 — chạy: python experiments/run_ablation.py --experiment E7")

## 3. E9 — NISQ noise robustness

In [ ]:
e9 = ablation.get("E9") or main_results.get("E9_noise") or {}
if e9:
    levels = sorted(float(k) for k in e9.keys() if e9[k].get("accuracy") is not None)
    accs = [e9[str(p)]["accuracy"] for p in levels]

    plt.figure(figsize=(9, 5))
    plt.plot(levels, accs, "o-", linewidth=2, markersize=9, color="#F44336")
    plt.xlabel("Depolarizing Noise Probability $p$")
    plt.ylabel("Test Accuracy")
    plt.title("NISQ Noise Robustness Analysis")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "noise_robustness.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Độ suy giảm tuyệt đối từ noiseless -> max noise
    if len(accs) >= 2:
        drop = accs[0] - accs[-1]
        print(f"Accuracy drop (p={levels[0]} -> {levels[-1]}): {drop:.4f}")
else:
    print("Chưa có dữ liệu E9 — chạy: python experiments/run_ablation.py --experiment E9")

## 4. t-SNE — Fused representations (quantum vs classical)

In [ ]:
# Trích xuất fused features từ model đã train để so sánh không gian biểu diễn
import torch
from types import SimpleNamespace

try:
    from src.training.train import load_config, build_dataloaders, build_model
    from sklearn.manifold import TSNE

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = load_config(PROJECT_ROOT / "experiments/configs/qfl_tensor.yaml")

    ckpt_path = PROJECT_ROOT / "checkpoints/best_multitask_latest.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Cần checkpoint tại {ckpt_path}")

    model = build_model(cfg, task="sentiment", model_name="qmmf").to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model_state_dict"])
    model.eval()

    loaders = build_dataloaders(cfg, "sentiment")
    feats, labels = [], []
    with torch.no_grad():
        n_collected = 0
        for batch in loaders["test"]:
            t_emb = model.text_encoder(
                batch["input_ids"].to(device), batch["attention_mask"].to(device))
            i_emb = model.image_encoder(batch["image"].to(device))
            half_t, half_i = t_emb.shape[-1] // 2, i_emb.shape[-1] // 2
            fused_q = model.shared_quantum.q_fusion(t_emb[:, :half_t], i_emb[:, :half_i])
            feats.append(fused_q.cpu())
            labels.append(batch["label"])
            n_collected += fused_q.size(0)
            if n_collected >= 1000:
                break

    X = torch.cat(feats).numpy()
    y = torch.cat(labels).numpy()

    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X2d = tsne.fit_transform(X)

    plt.figure(figsize=(9, 7))
    sc = plt.scatter(X2d[:, 0], X2d[:, 1], c=y, cmap="RdYlBu", alpha=0.7, s=25)
    plt.colorbar(sc, label="sentiment class")
    plt.title("t-SNE of Quantum Fused Representations")
    plt.xlabel("t-SNE dim 1")
    plt.ylabel("t-SNE dim 2")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tsne_fused.png", dpi=150, bbox_inches="tight")
    plt.show()
except FileNotFoundError as e:
    print("Bỏ qua t-SNE (cần checkpoint):", e)

## 5. Parameter efficiency & quantum circuit analysis (E10)

In [ ]:
e10 = ablation.get("E10") or main_results.get("E10_params") or {}
if e10:
    df = pd.DataFrame(e10).T
    display_cols = [c for c in ["total", "quantum", "classical", "ratio"] if c in df.columns]
    print(df[display_cols].to_string())

    try:
        from src.evaluation.visualize import plot_parameter_efficiency
        plot_parameter_efficiency(e10, save_path=str(FIG_DIR / "param_efficiency.png"))
        print("Saved:", FIG_DIR / "param_efficiency.png")
    except Exception as e:
        print("Plot failed:", e)
else:
    print("Chưa có dữ liệu E10 — chạy: python experiments/run_ablation.py --experiment E10")

In [ ]:
# Circuit resource summary cho paper (chạy được ngay, không cần checkpoint)
import pennylane as qml
import torch

from src.quantum.quantum_fusion import (
    _tensor_circuit,
    _attention_kernel,
    _interference_circuit,
)

summary_rows = []
for name, fn, args in [
    ("QFL-Tensor (8q, L3)", _tensor_circuit,
     (torch.randn(4), torch.randn(4), torch.randn(3, 16) * 0.01)),
    ("QFL-Attention kernel (4q)", _attention_kernel,
     (torch.randn(4), torch.randn(4), torch.randn(4) * 0.01)),
    ("QFL-Interference (8q)", _interference_circuit,
     (torch.randn(4), torch.randn(4), torch.randn(8) * 0.01)),
]:
    s = qml.specs(fn)(*args)
    gates = dict(s["resources"].gate_types)
    cnot = sum(v for g, v in gates.items() if "CNOT" in g or "cnot" in g.lower())
    rots = sum(v for g, v in gates.items() if any(r in g for r in ("RY", "RZ", "RX", "Rot")))
    summary_rows.append({
        "variant": name,
        "depth": s["resources"].depth,
        "total_gates": int(s["resources"].num_gates),
        "CNOT": cnot,
        "rotations": rots,
    })

circ_df = pd.DataFrame(summary_rows).set_index("variant")
print(circ_df.to_string())
circ_df.to_csv(PROJECT_ROOT / "experiments/results/circuit_resources.csv")